# Image Processing

This notebook introduces a compact image-processing workflow using NumPy, OpenCV, and Matplotlib. It uses a generated sample image, so it runs without downloading any external data.

## 1. Imports

OpenCV stores color images as BGR by default. Matplotlib expects RGB, so the helper below converts images before display.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


def show_image(title, image, cmap=None):
    plt.figure(figsize=(6, 4))
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 2. Create a Sample Image

A synthetic image is useful for teaching because the shapes, colors, and noise are controlled.

In [ ]:
height, width = 360, 540
image = np.zeros((height, width, 3), dtype=np.uint8)

# Smooth background gradient.
x = np.linspace(0, 255, width, dtype=np.uint8)
image[:, :, 0] = x
image[:, :, 1] = 80
image[:, :, 2] = 255 - x

# Add simple geometric objects.
cv2.rectangle(image, (60, 70), (230, 250), (40, 200, 80), thickness=-1)
cv2.circle(image, (375, 170), 85, (230, 170, 40), thickness=-1)
cv2.line(image, (40, 310), (500, 50), (255, 255, 255), thickness=8)

# Add repeatable noise.
rng = np.random.default_rng(7)
noise = rng.normal(0, 16, image.shape).astype(np.int16)
noisy_image = np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

show_image("Synthetic noisy image", noisy_image)

## 3. Inspect Pixels and Channels

Images are arrays. A color image has height, width, and channel dimensions.

In [ ]:
print(f"Shape: {noisy_image.shape}")
print(f"Data type: {noisy_image.dtype}")
print(f"Min pixel value: {noisy_image.min()}")
print(f"Max pixel value: {noisy_image.max()}")

blue, green, red = cv2.split(noisy_image)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, channel, title in zip(axes, [blue, green, red], ["Blue", "Green", "Red"]):
    ax.imshow(channel, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## 4. Convert to Grayscale

Many classic image-processing operations start with a single intensity channel.

In [ ]:
gray = cv2.cvtColor(noisy_image, cv2.COLOR_BGR2GRAY)
show_image("Grayscale", gray, cmap="gray")

## 5. Reduce Noise

Gaussian blur smooths local variation. It is often used before thresholding or edge detection.

In [ ]:
blurred = cv2.GaussianBlur(gray, (7, 7), sigmaX=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Original grayscale")
axes[0].axis("off")
axes[1].imshow(blurred, cmap="gray")
axes[1].set_title("Blurred")
axes[1].axis("off")
plt.tight_layout()

## 6. Threshold the Image

Thresholding converts a grayscale image into a binary mask. Otsu's method estimates a threshold from the image histogram.

In [ ]:
threshold_value, mask = cv2.threshold(
    blurred,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU,
)

print(f"Otsu threshold: {threshold_value:.1f}")
show_image("Binary mask", mask, cmap="gray")

## 7. Detect Edges

Canny edge detection finds strong intensity changes. It is sensitive to blur and threshold choices.

In [ ]:
edges = cv2.Canny(blurred, threshold1=60, threshold2=160)
show_image("Canny edges", edges, cmap="gray")

## 8. Save Results

Saving intermediate results makes notebooks easier to audit and reuse.

In [ ]:
output_dir = Path("output/notebooks/image_processing")
output_dir.mkdir(parents=True, exist_ok=True)

cv2.imwrite(str(output_dir / "sample_noisy.png"), noisy_image)
cv2.imwrite(str(output_dir / "grayscale.png"), gray)
cv2.imwrite(str(output_dir / "mask.png"), mask)
cv2.imwrite(str(output_dir / "edges.png"), edges)

sorted(path.name for path in output_dir.iterdir())

## Next Steps

Useful follow-up notebooks could cover histograms, morphology, contours, feature matching, camera calibration, and image segmentation.